# 04 · Introspection, `inspect`, `importlib` & Dynamic Code

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 50 min

### 🎯 What you will learn
- Reflection basics: `type`, `isinstance`, `dir`, `vars`, `getattr`/`setattr`/`hasattr`/`delattr`, `callable`
- The `inspect` module: members, **signatures**, source code, frames & the call stack
- Building tools with introspection (auto-CLI, dependency injection, validation)
- `importlib`: importing by name, reloading, finding modules, **plugin systems** with entry points
- `types`: `SimpleNamespace`, `MappingProxyType`, `ModuleType`, `FunctionType`
- `eval` / `exec` / `compile`: when (rarely) to use them, and the safe `ast.literal_eval`
- Useful `sys` internals

## 1. Reflection basics

In [1]:
class Robot:
    """A friendly robot."""
    model = "R2"
    def __init__(self, name):
        self.name = name
    def speak(self, words: str, times: int = 1) -> str:
        """Repeat words."""
        return " ".join([words] * times)

r = Robot("Robo")
print(type(r).__name__, isinstance(r, Robot), callable(r.speak), callable(r))
print(vars(r))                                      # instance namespace
print([a for a in dir(r) if not a.startswith("_")])
print(getattr(r, "name"), getattr(r, "missing", "default"))
setattr(r, "battery", 87)
print(hasattr(r, "battery"), r.battery)
method = getattr(r, "speak")                        # call a method chosen by NAME at runtime
print(method("hi", times=2))

Robot True True False
{'name': 'Robo'}
['model', 'name', 'speak']
Robo default
True 87
hi hi


### Dispatching by name: a tiny command interpreter

In [2]:
class Calculator:
    def cmd_add(self, a, b): return a + b
    def cmd_mul(self, a, b): return a * b

    def run(self, line):
        name, *args = line.split()
        handler = getattr(self, f"cmd_{name}", None)
        if handler is None:
            return f"unknown command {name!r}; try {[m[4:] for m in dir(self) if m.startswith('cmd_')]}"
        return handler(*map(float, args))

calc = Calculator()
print(calc.run("add 2 3"), calc.run("mul 4 5"), calc.run("pow 2 3"), sep=" | ")

5.0 | 20.0 | unknown command 'pow'; try ['add', 'mul']


## 2. The `inspect` module

In [3]:
import inspect

print(inspect.isclass(Robot), inspect.isfunction(Robot.speak), inspect.ismethod(r.speak))
print([name for name, _ in inspect.getmembers(Robot, inspect.isfunction)])
print(inspect.getdoc(Robot.speak))
print(inspect.getsource(Robot.speak))

True True True
['__init__', 'speak']
Repeat words.
    def speak(self, words: str, times: int = 1) -> str:
        """Repeat words."""
        return " ".join([words] * times)



### Signatures: the most useful part

In [4]:
sig = inspect.signature(Robot.speak)
print(sig)
for name, p in sig.parameters.items():
    print(f"  {name:<6} kind={p.kind.name:<22} default={p.default if p.default is not p.empty else '—':<5} "
          f"annotation={p.annotation if p.annotation is not p.empty else '—'}")
print("returns:", sig.return_annotation)

bound = sig.bind(r, "hello", times=3)          # check arguments WITHOUT calling the function
bound.apply_defaults()
print(bound.arguments)
try:
    sig.bind(r)                                 # missing 'words'
except TypeError as e:
    print("TypeError:", e)

(self, words: str, times: int = 1) -> str
  self   kind=POSITIONAL_OR_KEYWORD  default=—     annotation=—
  words  kind=POSITIONAL_OR_KEYWORD  default=—     annotation=<class 'str'>
  times  kind=POSITIONAL_OR_KEYWORD  default=1     annotation=<class 'int'>
returns: <class 'str'>
{'self': <__main__.Robot object at 0x10a351e80>, 'words': 'hello', 'times': 3}
TypeError: missing a required argument: 'words'


### Example: a runtime type-checking decorator built on signatures

In [5]:
import functools

def enforce_types(func):
    sig = inspect.signature(func)
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = sig.parameters[name].annotation
            if isinstance(expected, type) and not isinstance(value, expected):
                raise TypeError(f"{name}={value!r} is not {expected.__name__}")
        return func(*args, **kwargs)
    return wrapper

@enforce_types
def repeat(text: str, n: int) -> str:
    return text * n

print(repeat("ab", 3))
try:
    repeat("ab", "3")
except TypeError as e:
    print("TypeError:", e)

ababab
TypeError: n='3' is not int


This idea, reading type hints and signatures at runtime, is what **FastAPI**, **Typer**, **Pydantic's `validate_call`** and dependency-injection frameworks are built on.

### Frames and the call stack

In [6]:
def who_called_me():
    caller = inspect.stack()[1]
    return f"{caller.function}() at line {caller.lineno}"

def business_logic():
    return who_called_me()

print(business_logic())

def current_locals():
    frame = inspect.currentframe()
    try:
        return dict(frame.f_back.f_locals)      # the CALLER's local variables
    finally:
        del frame                               # avoid reference cycles

def demo():
    secret, answer = "xyz", 42
    return current_locals()

print(demo())

business_logic() at line 6
{'secret': 'xyz', 'answer': 42}


> ⚠️ Frame hacks are great for debuggers, loggers and test tools, but fragile in application code.

## 3. `importlib`: imports as functions

In [7]:
import importlib

json_mod = importlib.import_module("json")                     # import by string
print(json_mod.dumps({"dynamic": True}))
decoder = importlib.import_module("json.decoder")
print(decoder.JSONDecodeError)

spec = importlib.util.find_spec("requests")                    # is it installed? where?
print(spec.name if spec else None, spec.origin if spec else "not installed")
print(importlib.util.find_spec("surely_not_installed_pkg"))

{"dynamic": true}
<class 'json.decoder.JSONDecodeError'>
requests /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages/requests/__init__.py
None


### Loading a module from a file path, and reloading

In [8]:
import pathlib, sys, importlib.util

pathlib.Path("plugin_hello.py").write_text('VERSION = 1\ndef run():\n    return "hello v1"\n')
spec = importlib.util.spec_from_file_location("plugin_hello", "plugin_hello.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sys.modules["plugin_hello"] = mod
print(mod.run())

pathlib.Path("plugin_hello.py").write_text('VERSION = 2\ndef run():\n    return "hello v2 (reloaded)"\n')
importlib.invalidate_caches()
mod = importlib.reload(mod)
print(mod.run())

hello v1
hello v2 (reloaded)


### A plugin system with entry points

Installed packages can advertise plugins via `[project.entry-points."myapp.plugins"]` (Module 08 · 03). Your app discovers them without knowing their names in advance:

In [9]:
from importlib.metadata import entry_points

def discover(group):
    """Find plugins without importing them yet (importing is the slow/risky part)."""
    return {ep.name: ep for ep in entry_points(group=group)}

console = discover("console_scripts")
print(len(console), "console scripts registered, e.g.", sorted(console)[:5])
pytest_ep = console["pytest"]
print(pytest_ep.value, "→", pytest_ep.load())     # .load() imports the module and returns the object

71 console scripts registered, e.g. ['cffi-gen-src', 'coverage', 'coverage-3.15', 'coverage3', 'debugpy']
_pytest.config:_console_main → <function _console_main at 0x10a76fcc0>


Simpler alternative for in-project plugins: scan a package with `pkgutil.iter_modules()` and `importlib.import_module()`.

## 4. The `types` module

In [10]:
import types

cfg = types.SimpleNamespace(host="localhost", port=8080)     # quick attribute bag
cfg.debug = True
print(cfg, cfg.port)

frozen = types.MappingProxyType({"a": 1})                    # read-only dict VIEW
try:
    frozen["b"] = 2
except TypeError as e:
    print("TypeError:", e)

virtual = types.ModuleType("virtual", "a module created in memory")
virtual.answer = 42
sys.modules["virtual"] = virtual
import virtual as v
print(v.answer, v.__doc__)

print(type(len) is types.BuiltinFunctionType, type(repeat) is types.FunctionType)

namespace(host='localhost', port=8080, debug=True) 8080
TypeError: 'mappingproxy' object does not support item assignment
42 a module created in memory
True True


## 5. `eval`, `exec`, `compile`: handle with care

| Function | Runs | Returns |
|---|---|---|
| `eval(expr)` | a single **expression** | its value |
| `exec(code)` | any **statements** | `None` |
| `compile(src, filename, mode)` | turns source into a code object (`"eval"`, `"exec"`, `"single"`) | code object |

In [11]:
print(eval("2 ** 10 + len('abc')"))
namespace = {}
exec("def square(x):\n    return x * x\nresult = square(12)", namespace)
print(namespace["result"], namespace["square"](3))
code = compile("a + b", "<formula>", "eval")
print(eval(code, {"a": 2, "b": 5}))

1027
144 9
7


> ⚠️ **Never `eval`/`exec` untrusted input.** It can run any code: `__import__('os').system('rm -rf ~')`. Restricting `globals` is **not** a sandbox. To parse literal data (numbers, strings, lists, dicts) safely use **`ast.literal_eval`**:

In [12]:
import ast
print(ast.literal_eval("[1, 2, {'a': (3, 4)}]"))
try:
    ast.literal_eval("__import__('os').getcwd()")
except ValueError as e:
    print("ValueError: refused to evaluate a function call")

[1, 2, {'a': (3, 4)}]
ValueError: refused to evaluate a function call


Legitimate uses of `exec`/`compile`: code generators (like `dataclasses`, which builds `__init__` source and `exec`s it), templating engines, REPLs, plugin sandboxes in trusted environments.

## 6. Useful `sys` internals

In [13]:
print(sys.version_info[:3], sys.implementation.name, sys.platform)
print(sys.executable)
print(len(sys.modules), "modules loaded;", "json" in sys.modules)
print(sys.getrecursionlimit(), sys.maxsize, sys.byteorder)
print(sys.flags.optimize, sys.flags.utf8_mode)
print(sys.getrefcount(r) - 1, "references to r (lesson 05)")

(3, 15, 0) cpython darwin
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python
709 modules loaded; True
1000 9223372036854775807 little
0 1
3 references to r (lesson 05)


## 📝 Summary
- `getattr/setattr/hasattr/vars/dir` let you work with attributes by name.
- `inspect` gives signatures, members, source and frames: the foundation of many frameworks.
- `importlib` imports by name/path, reloads, and with `importlib.metadata.entry_points` powers plugin systems.
- `types` provides handy runtime objects; `eval`/`exec` are powerful and dangerous. Use `ast.literal_eval` for data.

## 🏋️ Practice
1. Write `describe(func)` that prints each parameter's name, kind, default and annotation.
2. Write `auto_cli(func)` that builds an `argparse` parser from a function's signature (required positionals, `--option` for defaults, `type` from annotations) and calls the function.
3. Write `load_all(package_name)` that imports every module in a package with `pkgutil.iter_modules`.

<details><summary>Show solution for 2</summary>

```python
import argparse, inspect

def auto_cli(func, argv=None):
    parser = argparse.ArgumentParser(description=func.__doc__)
    for name, p in inspect.signature(func).parameters.items():
        kind = p.annotation if p.annotation is not p.empty else str
        if p.default is p.empty:
            parser.add_argument(name, type=kind)
        else:
            parser.add_argument(f"--{name}", type=kind, default=p.default)
    return func(**vars(parser.parse_args(argv)))

def greet(name: str, times: int = 1):
    """Greet someone."""
    return " ".join([f"Hi {name}!"] * times)

print(auto_cli(greet, ["Asha", "--times", "2"]))
```
</details>